# 🎯 Attendance System - Preprocessing & Face Detection Pipeline

This notebook contains the specialized image preprocessing workflow for your facial recognition attendance system. It includes:

1. **Image Conversion** - Convert HEIC files and standardize formats
2. **Grayscale & CLAHE Preprocessing** - Apply grayscale conversion and CLAHE enhancement
3. **Face Detection & ROI Extraction** - Detect faces and extract ROI with padding
4. **Post-Processing** - Apply grayscale + CLAHE on cropped faces and resize to 128x128
5. **Data Augmentation (Member 3)** - Expand dataset using Albumentations + illumination evaluation
   - 5a: Raw image augmentation (for preview & illumination evaluation)
   - 5b: Face crop augmentation (training-ready augmented data)
6. **Training Data Preparation (Member 3)** - Load & split augmented face crops (80/20)
7. **LBPH Model Training & Evaluation (Member 3)** - Train LBPH recognizer, evaluate accuracy, compare with/without augmentation

## 📁 Expected Folder Structure
```
data/
├── raw/                     # Original images (input)
│   └── [person_name]/
├── augmented/               # Augmented face crops (training-ready)
│   └── [person_name]/
├── preprocessed/            # Grayscale + CLAHE processed images (Member 1 output)
│   └── [person_name]/
├── converted/               # Standardized JPG images
├── faces_detected/          # Extracted face ROIs (128x128)
├── final_processed/         # Post-processed face ROIs (Member 2 input)
└── preview/                 # Preview & evaluation results
    ├── augmentation_samples/
    └── evaluation_results/
models/
├── lbph_model.yml           # Trained LBPH face recognizer
└── label_map.txt            # Person name → label ID mapping
```


In [1]:
# Import all required libraries
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt
import face_recognition
import time
import random
from pathlib import Path
from PIL import Image, ImageEnhance, ImageOps
import dlib

# For HEIC conversion
try:
    from pillow_heif import register_heif_opener
    register_heif_opener()
    HAS_HEIF_SUPPORT = True
    print("✅ HEIC support available")
except ImportError:
    HAS_HEIF_SUPPORT = False
    print("⚠️  HEIC support not available. Install pillow-heif if needed.")

# For advanced augmentation
try:
    import albumentations as A
    HAS_ALBUMENTATIONS = True
    print("✅ Advanced augmentation available")
except ImportError:
    HAS_ALBUMENTATIONS = False
    print("⚠️  Advanced augmentation not available. Install albumentations if needed.")

print("📦 Core libraries loaded successfully!")

✅ HEIC support available
✅ Advanced augmentation available
📦 Core libraries loaded successfully!


# 🔧 Configuration Settings

Adjust these settings based on your needs:

In [ ]:
# === CONFIGURATION SETTINGS ===

# File paths (relative to this notebook)
DATA_FOLDER = os.path.join("..", "data")
RAW_FOLDER = os.path.join(DATA_FOLDER, "raw")
AUGMENTED_FOLDER = os.path.join(DATA_FOLDER, "augmented")
PREPROCESSED_FOLDER = os.path.join(DATA_FOLDER, "preprocessed")
CONVERTED_FOLDER = os.path.join(DATA_FOLDER, "converted")
FACES_DETECTED_FOLDER = os.path.join(DATA_FOLDER, "faces_detected")
FINAL_PROCESSED_FOLDER = os.path.join(DATA_FOLDER, "final_processed")
PREVIEW_FOLDER = os.path.join(DATA_FOLDER, "preview")
PREVIEW_AUGMENTATION_FOLDER = os.path.join(PREVIEW_FOLDER, "augmentation_samples")
PREVIEW_EVALUATION_FOLDER = os.path.join(PREVIEW_FOLDER, "evaluation_results")

# Models folder (at project root level, not inside data/)
MODELS_FOLDER = os.path.join("..", "models")

# Preprocessing settings
FACE_PADDING_RATIO = 0.2  # 20% padding around detected faces
OUTPUT_SIZE = (128, 128)  # Final size for preprocessed face images
CLAHE_CLIP_LIMIT = 2.0  # CLAHE clip limit for contrast enhancement
CLAHE_GRID_SIZE = (8, 8)  # CLAHE grid size for local enhancement

# Detection settings
DETECTION_SCALE_FACTOR = 0.25  # Scale factor for faster detection (1/4 size)
DETECTION_MODEL = "hog"  # Face detection model: 'hog' for speed, 'cnn' for accuracy

# Image conversion settings
IMAGE_QUALITY = 95  # JPG quality (1-100)
MAX_IMAGE_SIZE = (1024, 1024)  # Max size for converted images

# Augmentation settings (Member 3)
AUGMENTATION_TARGET_PER_PERSON = 100  # Target total images per person after augmentation
AUGMENTATION_BRIGHTNESS_LIMIT = 0.3   # Brightness variation range (±30%)
AUGMENTATION_CONTRAST_LIMIT = 0.3     # Contrast variation range (±30%)
AUGMENTATION_ROTATION_LIMIT = 15      # Max rotation angle in degrees
AUGMENTATION_NOISE_VAR_LIMIT = 25.0   # Gaussian noise variance limit
AUGMENTATION_GAMMA_RANGE = (0.5, 1.5) # Gamma correction range for illumination

# Display settings
PREVIEW_GRID_COLS = 4  # Columns in preview grids
FIGURE_SIZE = (15, 10)  # Default matplotlib figure size

print("⚙️ Configuration loaded!")
print(f"📁 Data folder:          {DATA_FOLDER}")
print(f"📁 Raw folder:           {RAW_FOLDER}")
print(f"📁 Augmented folder:     {AUGMENTED_FOLDER}")
print(f"📁 Preprocessed folder:  {PREPROCESSED_FOLDER}")
print(f"📁 Faces detected folder:{FACES_DETECTED_FOLDER}")
print(f"📁 Final processed:      {FINAL_PROCESSED_FOLDER}")
print(f"📁 Models folder:        {MODELS_FOLDER}")
print(f"🖼️  Output size:          {OUTPUT_SIZE}")
print(f"🔍 Detection model:      {DETECTION_MODEL}")
print(f"📊 CLAHE settings:       clip_limit={CLAHE_CLIP_LIMIT}, grid_size={CLAHE_GRID_SIZE}")
print(f"🔄 Augmentation target:  {AUGMENTATION_TARGET_PER_PERSON} images/person")


⚙️ Configuration loaded!
📁 Data folder: ..\data
🖼️  Face size: (224, 224)
🔄 Augmentations: 5 simple + 3 advanced


In [ ]:
# === CREATE DATASET FOLDERS ===
# Run this once to set up the full folder structure

people = [
    "zi_herng",
    "yong_kang",
    "yoke_hong",
    "xu_sheng",
    "xiang_yue",
    "wee_xuan",
    "shuang_quan",
    "qi_xuan",
    "marion",
    "kang_kai",
    "harry",
    "han_soon",
    "daniel",
    "chillien",
    "chern_tak",
    "benjamin",
]

# Folders that need per-person subfolders
person_folders = {
    "raw":          RAW_FOLDER,
    "augmented":    AUGMENTED_FOLDER,
    "preprocessed": PREPROCESSED_FOLDER,
}

# Standalone folders (no per-person subfolders needed)
standalone_folders = [
    CONVERTED_FOLDER,
    FACES_DETECTED_FOLDER,
    FINAL_PROCESSED_FOLDER,
    PREVIEW_AUGMENTATION_FOLDER,
    PREVIEW_EVALUATION_FOLDER,
    MODELS_FOLDER,
]

print("📁 Creating dataset folder structure...")
print("=" * 45)

# Create per-person subfolders
for folder_name, folder_path in person_folders.items():
    print(f"\n  📂 data/{folder_name}/")
    for person in people:
        person_path = os.path.join(folder_path, person)
        os.makedirs(person_path, exist_ok=True)
        print(f"    ✅ {person}/")

# Create standalone folders
print(f"\n  📂 Standalone folders:")
for folder_path in standalone_folders:
    os.makedirs(folder_path, exist_ok=True)
    display = os.path.relpath(folder_path, os.path.join("..", ".."))
    print(f"    ✅ {os.path.relpath(folder_path, '..')}")

print("\n✨ Done! All folders created.")
print(f"\n📸 Next step: Add each person's photos into their raw/ folder")
print(f"   Supported formats: .jpg, .jpeg, .png, .heic")
print(f"\n📂 Raw folder path: {os.path.abspath(RAW_FOLDER)}")


# 📊 Dataset Analysis

Let's first analyze your current dataset:

In [ ]:
def analyze_dataset():
    """Analyze the current dataset structure and contents"""
    
    if not os.path.exists(DATA_FOLDER):
        print("❌ Data folder not found!")
        return False
    
    print("🔍 DATASET ANALYSIS")
    print("=" * 50)
    
    # Get person folders (excluding processing folders)
    person_folders = [f for f in os.listdir(DATA_FOLDER) 
                     if os.path.isdir(os.path.join(DATA_FOLDER, f)) and 
                     f not in ['preprocessed', 'converted', 'faces_detected', 'preview', 'detectedFaces']]
    
    if not person_folders:
        print("No person folders found!")
        return False
    
    total_images = 0
    format_stats = {}
    person_stats = {}
    
    print(f"📁 Found {len(person_folders)} people:")
    
    for person in person_folders:
        person_path = os.path.join(DATA_FOLDER, person)
        
        # Count images
        all_files = [f for f in os.listdir(person_path) if os.path.isfile(os.path.join(person_path, f))]
        image_files = [f for f in all_files if Path(f).suffix.lower() in 
                      {'.jpg', '.jpeg', '.png', '.heic', '.JPG', '.JPEG', '.PNG', '.HEIC'}]
        
        person_stats[person] = len(image_files)
        total_images += len(image_files)
        
        # Count formats
        for img in image_files:
            ext = Path(img).suffix.lower()
            format_stats[ext] = format_stats.get(ext, 0) + 1
        
        print(f"  👤 {person}: {len(image_files)} images")
    
    print(f"\n📊 SUMMARY:")
    print(f"  • Total people: {len(person_folders)}")
    print(f"  • Total images: {total_images}")
    print(f"  • Average per person: {total_images/len(person_folders):.1f}")
    
    print(f"\n📸 IMAGE FORMATS:")
    for ext, count in sorted(format_stats.items()):
        percentage = (count/total_images) * 100
        print(f"  • {ext}: {count} ({percentage:.1f}%)")
    
    # Check processing folders
    print(f"\n📂 PROCESSING FOLDERS:")
    for folder_name in ['converted', 'preprocessed', 'faces_detected']:
        folder_path = os.path.join(DATA_FOLDER, folder_name)
        if os.path.exists(folder_path):
            files = sum([len(os.listdir(os.path.join(folder_path, p))) 
                        for p in os.listdir(folder_path) 
                        if os.path.isdir(os.path.join(folder_path, p))])
            print(f"  • {folder_name}: ✅ ({files} files)")
        else:
            print(f"  • {folder_name}: ❌ (not created)")
    
    return True

# Run the analysis
analyze_dataset()

🔍 DATASET ANALYSIS
📁 Found 16 people:
  👤 benjamin: 22 images
  👤 chern_tak: 36 images
  👤 chillien: 6 images
  👤 daniel: 54 images
  👤 han_soon: 3 images
  👤 harry: 23 images
  👤 kang_kai: 6 images
  👤 marion: 2 images
  👤 qi_xuan: 2 images
  👤 shuang_quan: 56 images
  👤 wee_xuan: 6 images
  👤 xiang_yue: 24 images
  👤 xu_sheng: 23 images
  👤 yoke_hong: 17 images
  👤 yong_kang: 3 images
  👤 zi_herng: 1 images

📊 SUMMARY:
  • Total people: 16
  • Total images: 284
  • Average per person: 17.8

📸 IMAGE FORMATS:
  • .heic: 173 (60.9%)
  • .jpeg: 21 (7.4%)
  • .jpg: 81 (28.5%)
  • .png: 9 (3.2%)

📂 PROCESSING FOLDERS:
  • converted: ✅ (0 files)
  • processed: ✅ (0 files)
  • augmented: ✅ (0 files)


True

# 🔄 Step 1: Image Conversion & Standardization

Convert HEIC files to JPG and standardize all images:

In [4]:
def convert_heic_to_jpg(input_path, output_path, quality=95):
    """Convert HEIC file to JPG"""
    if not HAS_HEIF_SUPPORT:
        print(f"⚠️ HEIC support not available. Skipping {input_path}")
        return False
    
    try:
        with Image.open(input_path) as image:
            if image.mode != 'RGB':
                image = image.convert('RGB')
            image.save(output_path, 'JPEG', quality=quality, optimize=True)
            return True
    except Exception as e:
        print(f"❌ Error converting {input_path}: {e}")
        return False

def standardize_image(input_path, output_path, target_size=MAX_IMAGE_SIZE, quality=IMAGE_QUALITY):
    """Standardize image size and format"""
    try:
        with Image.open(input_path) as image:
            if image.mode != 'RGB':
                image = image.convert('RGB')
            
            # Resize while maintaining aspect ratio
            image = ImageOps.fit(image, target_size, Image.Resampling.LANCZOS)
            image.save(output_path, 'JPEG', quality=quality, optimize=True)
            return True
    except Exception as e:
        print(f"❌ Error standardizing {input_path}: {e}")
        return False

def convert_all_images():
    """Convert and standardize all images"""
    print("🔄 STARTING IMAGE CONVERSION")
    print("=" * 50)
    
    # Create output folder
    os.makedirs(CONVERTED_FOLDER, exist_ok=True)
    
    # Get person folders
    person_folders = [f for f in os.listdir(DATA_FOLDER) 
                     if os.path.isdir(os.path.join(DATA_FOLDER, f)) and 
                     f not in ['processed', 'converted', 'augmented', 'preview', 'detectedFaces']]
    
    total_converted = 0
    total_failed = 0
    
    for person in person_folders:
        person_input_path = os.path.join(DATA_FOLDER, person)
        person_output_path = os.path.join(CONVERTED_FOLDER, person)
        os.makedirs(person_output_path, exist_ok=True)
        
        # Get image files
        image_extensions = {'.jpg', '.jpeg', '.png', '.heic', '.JPG', '.JPEG', '.PNG', '.HEIC'}
        image_files = [f for f in os.listdir(person_input_path) 
                      if Path(f).suffix in image_extensions]
        
        print(f"\n👤 Processing {person} ({len(image_files)} images):")
        
        for i, filename in enumerate(image_files):
            input_file = os.path.join(person_input_path, filename)
            output_filename = f"{person}_{i+1:03d}.jpg"
            output_file = os.path.join(person_output_path, output_filename)
            
            # Convert based on file type
            if Path(filename).suffix.lower() == '.heic':
                success = convert_heic_to_jpg(input_file, output_file, IMAGE_QUALITY)
            else:
                success = standardize_image(input_file, output_file)
            
            if success:
                print(f"  ✅ {filename} → {output_filename}")
                total_converted += 1
            else:
                print(f"  ❌ Failed: {filename}")
                total_failed += 1
    
    print(f"\n✨ CONVERSION COMPLETE!")
    print(f"  ✅ Converted: {total_converted}")
    print(f"  ❌ Failed: {total_failed}")
    print(f"  📁 Output: {CONVERTED_FOLDER}")
    
    return total_converted, total_failed

# Run conversion
conversion_stats = convert_all_images()

🔄 STARTING IMAGE CONVERSION

👤 Processing benjamin (22 images):
  ✅ IMG_1924.JPG → benjamin_001.jpg
  ✅ IMG_1925.JPG → benjamin_002.jpg
  ✅ IMG_1926.JPG → benjamin_003.jpg
  ✅ IMG_1929.JPG → benjamin_004.jpg
  ✅ IMG_1930.JPG → benjamin_005.jpg
  ✅ IMG_1931.JPG → benjamin_006.jpg
  ✅ IMG_1932.JPG → benjamin_007.jpg
  ✅ IMG_1933.JPG → benjamin_008.jpg
  ✅ IMG_1934.JPG → benjamin_009.jpg
  ✅ IMG_8773.HEIC → benjamin_010.jpg
  ✅ IMG_8774.HEIC → benjamin_011.jpg
  ✅ IMG_8775.HEIC → benjamin_012.jpg
  ✅ IMG_E1924.JPG → benjamin_013.jpg
  ✅ IMG_E1925.JPG → benjamin_014.jpg
  ✅ IMG_E1926.JPG → benjamin_015.jpg
  ✅ IMG_E1929.JPG → benjamin_016.jpg
  ✅ IMG_E1930.JPG → benjamin_017.jpg
  ✅ IMG_E1931.JPG → benjamin_018.jpg
  ✅ IMG_E1932.JPG → benjamin_019.jpg
  ✅ IMG_E1933.JPG → benjamin_020.jpg
  ✅ IMG_E1934.JPG → benjamin_021.jpg
  ✅ IMG_E8773.HEIC → benjamin_022.jpg

👤 Processing chern_tak (36 images):
  ✅ BFOQ6507.JPG → chern_tak_001.jpg
  ✅ GGDI6673.JPG → chern_tak_002.jpg
  ✅ IJES7702.JPG → 

# 🔄 Step 2: Grayscale & CLAHE Preprocessing

Apply grayscale conversion and CLAHE enhancement to all converted images for better face detection:

In [ ]:
class ImagePreprocessor:
    def __init__(self, clahe_clip_limit=CLAHE_CLIP_LIMIT, clahe_grid_size=CLAHE_GRID_SIZE):
        """Initialize preprocessor with CLAHE settings"""
        self.clahe = cv2.createCLAHE(clipLimit=clahe_clip_limit, tileGridSize=clahe_grid_size)
        print(f"✅ Image preprocessor initialized")
        print(f"   • CLAHE clip limit: {clahe_clip_limit}")
        print(f"   • CLAHE grid size: {clahe_grid_size}")
    
    def preprocess_image(self, image):
        """Apply grayscale conversion and CLAHE enhancement"""
        # Convert to grayscale
        if len(image.shape) == 3:
            gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
        else:
            gray = image
        
        # Apply CLAHE enhancement
        enhanced = self.clahe.apply(gray)
        
        return gray, enhanced
    
    def preprocess_folder(self, input_folder, output_folder):
        """Process all images in a folder with grayscale + CLAHE"""
        # Create output folder
        os.makedirs(output_folder, exist_ok=True)
        
        # Get person folders
        person_folders = [f for f in os.listdir(input_folder) 
                         if os.path.isdir(os.path.join(input_folder, f))]
        
        total_processed = 0
        total_failed = 0
        
        for person in person_folders:
            person_input_path = os.path.join(input_folder, person)
            person_output_path = os.path.join(output_folder, person)
            os.makedirs(person_output_path, exist_ok=True)
            
            # Get image files
            image_files = [f for f in os.listdir(person_input_path) if f.lower().endswith('.jpg')]
            
            print(f"\n👤 Preprocessing {person} ({len(image_files)} images):")
            
            for image_file in image_files:
                input_path = os.path.join(person_input_path, image_file)
                image = cv2.imread(input_path)
                
                if image is None:
                    print(f"  ❌ Failed to read: {image_file}")
                    total_failed += 1
                    continue
                
                # Apply preprocessing
                gray, enhanced = self.preprocess_image(image)
                
                # Save both versions
                base_name = Path(image_file).stem
                
                # Save grayscale version
                gray_path = os.path.join(person_output_path, f"{base_name}_gray.jpg")
                cv2.imwrite(gray_path, gray)
                
                # Save CLAHE enhanced version
                enhanced_path = os.path.join(person_output_path, f"{base_name}_clahe.jpg")
                cv2.imwrite(enhanced_path, enhanced)
                
                total_processed += 1
                print(f"  ✅ {image_file} → gray + CLAHE versions")
        
        print(f"\n✨ PREPROCESSING COMPLETE!")
        print(f"  ✅ Processed: {total_processed}")
        print(f"  ❌ Failed: {total_failed}")
        print(f"  📁 Output: {output_folder}")
        
        return total_processed, total_failed

# Initialize preprocessor
preprocessor = ImagePreprocessor()
print("🔄 Image preprocessor ready!")

✅ Dlib face detector loaded
🔍 Face detector initialized!


In [ ]:
def run_preprocessing():
    """Apply grayscale + CLAHE preprocessing to converted images"""
    print("🔄 STARTING PREPROCESSING")  
    print("=" * 50)
    
    if not os.path.exists(CONVERTED_FOLDER):
        print("❌ Converted folder not found! Please run image conversion first.")
        return
    
    # Run preprocessing
    stats = preprocessor.preprocess_folder(CONVERTED_FOLDER, PREPROCESSED_FOLDER)
    
    print("\n📊 PREPROCESSING SUMMARY:")
    print(f"Images processed: {stats[0]}")
    print(f"Failed: {stats[1]}")
    print(f"📁 Output: {PREPROCESSED_FOLDER}")
    
    return stats

# Run preprocessing
preprocessing_stats = run_preprocessing()

👁️ STARTING FACE EXTRACTION

👤 Processing benjamin (22 images):
  ⚠️ No face: benjamin_001.jpg
  ⚠️ No face: benjamin_002.jpg
  ⚠️ No face: benjamin_003.jpg
  ⚠️ No face: benjamin_004.jpg
  ⚠️ No face: benjamin_005.jpg
  ⚠️ No face: benjamin_006.jpg
  ⚠️ No face: benjamin_007.jpg
  ⚠️ No face: benjamin_008.jpg
  ⚠️ No face: benjamin_009.jpg
  ⚠️ No face: benjamin_010.jpg
  ✅ benjamin_011.jpg → benjamin_011_face.jpg
  ✅ benjamin_012.jpg → benjamin_012_face.jpg
  ⚠️ No face: benjamin_013.jpg
  ⚠️ No face: benjamin_014.jpg
  ⚠️ No face: benjamin_015.jpg
  ⚠️ No face: benjamin_016.jpg
  ⚠️ No face: benjamin_017.jpg
  ✅ benjamin_018.jpg → benjamin_018_face.jpg
  ✅ benjamin_019.jpg → benjamin_019_face.jpg
  ⚠️ No face: benjamin_020.jpg
  ✅ benjamin_021.jpg → benjamin_021_face.jpg
  ⚠️ No face: benjamin_022.jpg
  📊 benjamin: 5 faces extracted

👤 Processing chern_tak (36 images):
  ⚠️ No face: chern_tak_001.jpg
  ⚠️ No face: chern_tak_002.jpg
  ⚠️ No face: chern_tak_003.jpg
  ✅ chern_tak_004.j

KeyboardInterrupt: 

# 👁️ Step 3: Face Detection & ROI Extraction

Detect faces in preprocessed images and extract regions of interest with padding:

In [ ]:
class FaceROIExtractor:
    def __init__(self, padding_ratio=FACE_PADDING_RATIO, target_size=OUTPUT_SIZE):
        """Initialize face ROI extractor for preprocessed images"""
        self.padding_ratio = padding_ratio
        self.target_size = target_size
        
        # For OpenCV cascade classifier
        self.face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_frontalface_default.xml')
        
        print(f"✅ Face ROI extractor initialized")
        print(f"   • Padding ratio: {padding_ratio}")
        print(f"   • Target output size: {target_size}")
    
    def detect_faces_opencv(self, image):
        """Detect faces using OpenCV cascade classifier"""
        faces = self.face_cascade.detectMultiScale(
            image,
            scaleFactor=1.1,
            minNeighbors=5,
            minSize=(30, 30)
        )
        return [(x, y, w, h) for (x, y, w, h) in faces]
    
    def detect_faces_dlib(self, image):
        """Detect faces using face_recognition library (HOG)"""
        try:
            # face_recognition expects RGB format
            if len(image.shape) == 3:
                rgb_image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
            else:
                rgb_image = cv2.cvtColor(image, cv2.COLOR_GRAY2RGB)
            
            # Get face locations
            face_locations = face_recognition.face_locations(rgb_image, model="hog")
            
            # Convert to (x, y, w, h) format
            faces = []
            for (top, right, bottom, left) in face_locations:
                x, y, w, h = left, top, right - left, bottom - top
                faces.append((x, y, w, h))
            
            return faces
        except Exception as e:
            print(f"  ⚠️ HOG detection failed, falling back to OpenCV: {e}")
            return self.detect_faces_opencv(image)
    
    def extract_roi_with_padding(self, image, x, y, w, h):
        """Extract face ROI with padding"""
        # Calculate padding
        pad_w = int(w * self.padding_ratio)
        pad_h = int(h * self.padding_ratio)
        
        # Calculate padded coordinates
        x1 = max(0, x - pad_w)
        y1 = max(0, y - pad_h)
        x2 = min(image.shape[1], x + w + pad_w)
        y2 = min(image.shape[0], y + h + pad_h)
        
        # Extract ROI
        roi = image[y1:y2, x1:x2]
        
        return roi, (x1, y1, x2, y2)
    
    def process_preprocessed_folder(self, input_folder, output_folder):
        """Process preprocessed images and extract face ROIs"""
        os.makedirs(output_folder, exist_ok=True)
        
        # Get person folders  
        person_folders = [f for f in os.listdir(input_folder) 
                         if os.path.isdir(os.path.join(input_folder, f))]
        
        total_processed = 0
        total_faces = 0
        total_failed = 0
        
        for person in person_folders:
            person_input_path = os.path.join(input_folder, person)
            person_output_path = os.path.join(output_folder, person)
            os.makedirs(person_output_path, exist_ok=True)
            
            # Get preprocessed images (both gray and clahe)
            image_files = [f for f in os.listdir(person_input_path) 
                          if f.lower().endswith('.jpg') and ('_clahe.jpg' in f or '_gray.jpg' in f)]
            
            print(f"\n👤 Processing {person} ({len(image_files)} preprocessed images):")
            
            for image_file in image_files:
                input_path = os.path.join(person_input_path, image_file)
                image = cv2.imread(input_path)
                
                if image is None:
                    print(f"  ❌ Failed to read: {image_file}")
                    total_failed += 1
                    continue
                
                # Convert to grayscale for face detection if needed
                if len(image.shape) == 3:
                    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
                else:
                    gray = image
                
                # Detect faces
                faces = self.detect_faces_dlib(gray)
                
                if not faces:
                    print(f"  ❌ No face detected: {image_file}")
                    total_failed += 1
                    continue
                
                # Process each detected face
                base_name = Path(image_file).stem
                for i, (x, y, w, h) in enumerate(faces):
                    # Extract ROI with padding
                    roi, coords = self.extract_roi_with_padding(gray, x, y, w, h)
                    
                    # Resize to target size
                    resized_roi = cv2.resize(roi, self.target_size)
                    
                    # Save face ROI
                    face_filename = f"{base_name}_face_{i}.jpg"
                    face_path = os.path.join(person_output_path, face_filename)
                    cv2.imwrite(face_path, resized_roi, [cv2.IMWRITE_JPEG_QUALITY, IMAGE_QUALITY])
                    
                    total_faces += 1
                    print(f"  ✅ {image_file} → face_{i} ({self.target_size}px)")
                
                total_processed += 1
        
        print(f"\n✨ FACE ROI EXTRACTION COMPLETE!")
        print(f"  ✅ Images processed: {total_processed}")
        print(f"  👤 Faces extracted: {total_faces}")
        print(f"  ❌ Failed: {total_failed}")
        print(f"  📁 Output: {output_folder}")
        
        return total_processed, total_faces, total_failed

# Initialize face ROI extractor
roi_extractor = FaceROIExtractor()
print("👁️ Face ROI extractor ready!")

In [ ]:
def extract_faces_from_preprocessed():
    """Extract face ROIs from preprocessed images"""
    print("👁️ STARTING FACE ROI EXTRACTION")
    print("=" * 50)
    
    if not os.path.exists(PREPROCESSED_FOLDER):
        print("❌ Preprocessed folder not found! Please run preprocessing first.")
        return
    
    # Extract faces from preprocessed images
    stats = roi_extractor.process_preprocessed_folder(PREPROCESSED_FOLDER, FACES_DETECTED_FOLDER)
    
    print("\n📊 EXTRACTION SUMMARY:")
    print(f"Images processed: {stats[0]}")
    print(f"Faces extracted: {stats[1]}")
    print(f"Failed: {stats[2]}")
    
    return stats

# Run face ROI extraction
face_roi_stats = extract_faces_from_preprocessed()

# ⚡ Step 4: Post-process Extracted Faces

Apply final grayscale conversion and CLAHE enhancement to extracted faces for optimal quality:

In [ ]:
class FacePostProcessor:
    def __init__(self, clahe_clip_limit=CLAHE_CLIP_LIMIT, clahe_grid_size=CLAHE_GRID_SIZE, target_size=OUTPUT_SIZE):
        """Initialize post-processor for extracted faces"""
        self.clahe = cv2.createCLAHE(clipLimit=clahe_clip_limit, tileGridSize=clahe_grid_size)
        self.target_size = target_size
        
        print(f"✅ Face post-processor initialized")
        print(f"   • Target size: {target_size}")
        print(f"   • CLAHE clip limit: {clahe_clip_limit}")
        print(f"   • CLAHE grid size: {clahe_grid_size}")
    
    def post_process_face(self, face_image):
        """Apply grayscale conversion and CLAHE to an extracted face"""
        # Convert to grayscale if needed
        if len(face_image.shape) == 3:
            gray = cv2.cvtColor(face_image, cv2.COLOR_BGR2GRAY)
        else:
            gray = face_image
        
        # Apply CLAHE enhancement
        enhanced = self.clahe.apply(gray)
        
        # Resize to target size
        if enhanced.shape[:2] != self.target_size:
            enhanced = cv2.resize(enhanced, self.target_size)
        
        return enhanced
    
    def post_process_folder(self, input_folder, output_folder):
        """Post-process all extracted faces"""
        os.makedirs(output_folder, exist_ok=True)
        
        # Get person folders
        person_folders = [f for f in os.listdir(input_folder) 
                         if os.path.isdir(os.path.join(input_folder, f))]
        
        total_processed = 0
        total_failed = 0
        
        for person in person_folders:
            person_input_path = os.path.join(input_folder, person)
            person_output_path = os.path.join(output_folder, person)
            os.makedirs(person_output_path, exist_ok=True)
            
            # Get face images
            face_files = [f for f in os.listdir(person_input_path) 
                         if f.lower().endswith('.jpg') and '_face_' in f]
            
            print(f"\n👤 Post-processing {person} ({len(face_files)} faces):")
            
            for face_file in face_files:
                input_path = os.path.join(person_input_path, face_file)
                face_image = cv2.imread(input_path, cv2.IMREAD_GRAYSCALE)
                
                if face_image is None:
                    print(f"  ❌ Failed to read: {face_file}")
                    total_failed += 1
                    continue
                
                # Post-process the face
                processed_face = self.post_process_face(face_image)
                
                # Save processed face
                base_name = Path(face_file).stem
                output_filename = f"{base_name}_processed.jpg"
                output_path = os.path.join(person_output_path, output_filename)
                cv2.imwrite(output_path, processed_face, [cv2.IMWRITE_JPEG_QUALITY, IMAGE_QUALITY])
                
                total_processed += 1
                print(f"  ✅ {face_file} → processed ({self.target_size}px)")
        
        print(f"\n✨ POST-PROCESSING COMPLETE!")
        print(f"  ✅ Faces processed: {total_processed}")
        print(f"  ❌ Failed: {total_failed}")
        print(f"  📁 Output: {output_folder}")
        
        return total_processed, total_failed

# Initialize post-processor
face_post_processor = FacePostProcessor()
print("⚡ Face post-processor ready!")

In [ ]:
def post_process_extracted_faces():
    """Apply final post-processing to extracted face ROIs"""
    print("⚡ STARTING FACE POST-PROCESSING")
    print("=" * 50)
    
    if not os.path.exists(FACES_DETECTED_FOLDER):
        print("❌ Face detection folder not found! Please run face extraction first.")
        return
    
    # Post-process all extracted faces
    stats = face_post_processor.post_process_folder(FACES_DETECTED_FOLDER, FINAL_PROCESSED_FOLDER)
    
    print("\n📊 POST-PROCESSING SUMMARY:")
    print(f"Faces processed: {stats[0]}")
    print(f"Failed: {stats[1]}")
    print(f"📁 Final output: {FINAL_PROCESSED_FOLDER}")
    
    return stats

# Run face post-processing
post_processing_stats = post_process_extracted_faces()

# 📊 Processing Summary & Statistics

View the complete processing statistics:

In [ ]:
def generate_processing_summary():
    """Generate comprehensive processing summary across all pipeline stages."""
    
    print("📊 COMPLETE PROCESSING SUMMARY")
    print("=" * 60)
    
    # Helper: count images in a folder (recursively through person subfolders)
    def count_folder_images(folder_path):
        if not os.path.exists(folder_path):
            return 0, 0  # (people_count, image_count)
        person_dirs = [d for d in os.listdir(folder_path) 
                      if os.path.isdir(os.path.join(folder_path, d))]
        total_files = 0
        for person in person_dirs:
            person_path = os.path.join(folder_path, person)
            files = [f for f in os.listdir(person_path) 
                    if Path(f).suffix.lower() in {'.jpg', '.jpeg', '.png'}]
            total_files += len(files)
        return len(person_dirs), total_files
    
    # Original raw dataset
    raw_people, raw_total = count_folder_images(RAW_FOLDER)
    print(f"\n📷 RAW DATASET:")
    print(f"  • People: {raw_people}")
    print(f"  • Images: {raw_total}")
    if raw_people > 0:
        print(f"  • Avg per person: {raw_total / raw_people:.1f}")
    
    # Processing stages
    stages = [
        ("🔄 CONVERTED",       CONVERTED_FOLDER,       None),
        ("🔄 AUGMENTED",       AUGMENTED_FOLDER,        "augmentation"),
        ("🔄 PREPROCESSED",    PREPROCESSED_FOLDER,     None),
        ("👤 FACES DETECTED",  FACES_DETECTED_FOLDER,   "detection"),
        ("⚡ FINAL PROCESSED", FINAL_PROCESSED_FOLDER,  None),
    ]
    
    for stage_name, folder_path, stage_type in stages:
        people_count, file_count = count_folder_images(folder_path)
        
        if os.path.exists(folder_path):
            print(f"\n{stage_name}:")
            print(f"  • People: {people_count}")
            print(f"  • Images: {file_count}")
            
            if stage_type == "augmentation" and raw_total > 0 and file_count > 0:
                multiplication = file_count / raw_total
                print(f"  • Dataset multiplication: {multiplication:.1f}x")
                if people_count > 0:
                    print(f"  • Avg per person: {file_count / people_count:.1f}")
            
            if stage_type == "detection" and raw_total > 0 and file_count > 0:
                print(f"  • Faces per original: {file_count / raw_total:.1f}")
        else:
            print(f"\n❌ {stage_name}: Folder not created yet")
    
    print(f"\n{'=' * 60}")
    print(f"✨ Summary complete!")

generate_processing_summary()

# 📹 Step 5: Live Face Detection (Updated)

Enhanced real-time face detection with the new pipeline:

In [ ]:
def start_live_face_detection():
    """Start live face detection with pipeline integration"""
    
    print("📹 STARTING LIVE FACE DETECTION")
    print("Controls:")
    print("  • Press 'q' to quit")
    print("  • Press 's' to save detected face")
    print("  • Press 'p' to toggle preprocessing display")
    
    # Setup
    video_capture = cv2.VideoCapture(0)
    if not video_capture.isOpened():
        print("❌ Could not open camera!")
        return
    
    # Create output folder for detected faces
    detected_faces_folder = os.path.join(DATA_FOLDER, "detectedFaces")
    os.makedirs(detected_faces_folder, exist_ok=True)
    
    # Settings
    last_save = 0
    save_delay = 2
    show_preprocessing = False
    
    # CLAHE for preprocessing
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    
    print("🎥 Camera active. Press 'q' to stop.")
    
    while True:
        ret, frame = video_capture.read()
        if not ret:
            break
        
        # Create display frame
        display_frame = frame.copy()
        
        # Preprocessing for better detection
        small_frame = cv2.resize(frame, (0, 0), fx=0.25, fy=0.25)
        gray_small = cv2.cvtColor(small_frame, cv2.COLOR_BGR2GRAY)
        equ_small = clahe.apply(gray_small)
        
        # Face detection using our pipeline
        face_locations = face_recognition.face_locations(equ_small, model="hog")
        
        for face_loc in face_locations:
            # Scale coordinates back to original frame size (x4)
            top, right, bottom, left = [v * 4 for v in face_loc]
            
            # Apply padding
            h, w = bottom - top, right - left
            pad_h, pad_w = int(h * FACE_PADDING_RATIO), int(w * FACE_PADDING_RATIO)
            
            # Boundary handling
            img_h, img_w = frame.shape[:2]
            p_top = max(0, top - pad_h)
            p_bottom = min(img_h, bottom + pad_h)
            p_left = max(0, left - pad_w)
            p_right = min(img_w, right + pad_w)
            
            # Save face if enough time has passed
            if time.time() - last_save > save_delay:
                face_crop = frame[p_top:p_bottom, p_left:p_right]
                
                # Resize to standard size
                if face_crop.size > 0:
                    face_resized = cv2.resize(face_crop, OUTPUT_SIZE, interpolation=cv2.INTER_AREA)
                    
                    timestamp = time.strftime('%Y%m%d_%H%M%S')
                    filename = f"live_face_{timestamp}.jpg"
                    save_path = os.path.join(detected_faces_folder, filename)
                    
                    cv2.imwrite(save_path, face_resized)
                    print(f"💾 Saved: {filename}")
                    last_save = time.time()
            
            # Draw detection rectangles
            cv2.rectangle(display_frame, (left, top), (right, bottom), (0, 255, 0), 2)  # Face
            cv2.rectangle(display_frame, (p_left, p_top), (p_right, p_bottom), (255, 0, 0), 1)  # Padding
            
            # Add label
            cv2.putText(display_frame, f"Face {OUTPUT_SIZE[0]}x{OUTPUT_SIZE[1]}", 
                       (left, top - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 1)
        
        # Add instructions
        cv2.putText(display_frame, "Press 'q' to quit, 's' to save, 'p' for preprocessing", 
                   (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 1)
        
        cv2.putText(display_frame, f"Faces: {len(face_locations)} | Saved to: detectedFaces/", 
                   (10, 60), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1)
        
        # Show preprocessing if enabled
        if show_preprocessing:
            # Show preprocessing steps side by side
            gray_display = cv2.cvtColor(cv2.resize(equ_small, (frame.shape[1]//4, frame.shape[0]//4)), cv2.COLOR_GRAY2BGR)
            display_frame[10:10+gray_display.shape[0], -gray_display.shape[1]-10:-10] = gray_display
            cv2.putText(display_frame, "Preprocessing", 
                       (display_frame.shape[1]-gray_display.shape[1]-10, gray_display.shape[0]+25), 
                       cv2.FONT_HERSHEY_SIMPLEX, 0.4, (0, 255, 255), 1)
        
        cv2.imshow('Attendance System - Enhanced Face Detection', display_frame)
        
        # Handle key presses
        key = cv2.waitKey(1) & 0xFF
        if key == ord('q'):
            break
        elif key == ord('s'):
            print("Manual save triggered!")
            last_save = 0  # Force save on next detection
        elif key == ord('p'):
            show_preprocessing = not show_preprocessing
            print(f"Preprocessing display: {'ON' if show_preprocessing else 'OFF'}")
    
    video_capture.release()
    cv2.destroyAllWindows()
    print("📹 Live detection stopped")

print("📹 Live face detection ready!")
print("Run the cell below to start the camera...")

In [ ]:
# Uncomment the line below to start live face detection
# start_live_face_detection()

print("💡 Uncomment the line above to start live face detection")
print("⚠️  Make sure you have a working camera connected")

# 🛠️ Utility Functions

Additional helper functions for your workflow:

In [ ]:
def preview_random_augmentations(person_name, num_examples=1):
    """Show random augmentation examples for a person using the new DataAugmentor."""
    
    person_path = os.path.join(AUGMENTED_FOLDER, person_name)
    if not os.path.exists(person_path):
        # Fallback to raw folder
        person_path = os.path.join(RAW_FOLDER, person_name)
        if not os.path.exists(person_path):
            print(f"❌ No images found for {person_name}")
            return
    
    image_files = [f for f in os.listdir(person_path) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
    if not image_files:
        print(f"❌ No image files found for {person_name}")
        return
    
    # Take random sample
    sample_files = random.sample(image_files, min(num_examples, len(image_files)))
    
    for img_file in sample_files:
        img_path = os.path.join(person_path, img_file)
        image = cv2.imread(img_path)
        
        if image is None:
            continue
        
        # Generate augmentations using the DataAugmentor
        pipeline_names = list(augmentor.pipelines.keys())
        aug_images = [augmentor.augment_single_image(image, p) for p in pipeline_names]
        
        # Create visualization
        all_images = [image] + aug_images
        titles = ["Original"] + pipeline_names
        
        cols = len(all_images)
        fig, axes = plt.subplots(1, cols, figsize=(3 * cols, 3.5))
        for i, (img, title) in enumerate(zip(all_images, titles)):
            axes[i].imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
            axes[i].set_title(title, fontsize=10)
            axes[i].axis('off')
        
        plt.suptitle(f"Augmentation Examples - {person_name} - {img_file}", fontsize=12)
        plt.tight_layout()
        plt.show()

def clean_processing_folders():
    """Clean all processing output folders (keeps raw/)"""
    import shutil
    
    folders_to_clean = [
        CONVERTED_FOLDER, PREPROCESSED_FOLDER, AUGMENTED_FOLDER,
        FACES_DETECTED_FOLDER, FINAL_PROCESSED_FOLDER, PREVIEW_FOLDER
    ]
    
    print("🧹 CLEANING PROCESSING FOLDERS")
    
    for folder in folders_to_clean:
        if os.path.exists(folder):
            try:
                shutil.rmtree(folder)
                print(f"✅ Cleaned: {os.path.relpath(folder, '..')}")
            except Exception as e:
                print(f"❌ Error cleaning {folder}: {e}")
        else:
            print(f"⚠️ Not found: {os.path.relpath(folder, '..')}")
    
    print("🧹 Cleaning complete!")

def export_face_encodings(output_path="face_encodings.npz"):
    """Export face encodings for all final processed faces"""
    
    source_folder = FINAL_PROCESSED_FOLDER
    if not os.path.exists(source_folder):
        source_folder = FACES_DETECTED_FOLDER
        if not os.path.exists(source_folder):
            print("❌ No processed faces found! Run the pipeline first.")
            return
    
    print(f"🔍 Extracting face encodings from {os.path.relpath(source_folder, '..')}...")
    
    encodings = []
    labels = []
    
    person_folders = [f for f in os.listdir(source_folder) 
                     if os.path.isdir(os.path.join(source_folder, f))]
    
    for person in person_folders:
        person_path = os.path.join(source_folder, person)
        face_files = [f for f in os.listdir(person_path) if f.lower().endswith('.jpg')]
        
        print(f"  👤 {person}: {len(face_files)} faces")
        
        for face_file in face_files:
            face_path = os.path.join(person_path, face_file)
            image = cv2.imread(face_path)
            
            if image is None:
                continue
            
            # Convert to RGB
            rgb_image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
            
            # Get encoding
            face_encs = face_recognition.face_encodings(rgb_image)
            
            if face_encs:
                encodings.append(face_encs[0])
                labels.append(person)
    
    if encodings:
        np.savez(output_path, encodings=encodings, labels=labels)
        print(f"💾 Saved {len(encodings)} face encodings to {output_path}")
    else:
        print("❌ No face encodings found!")

print("🛠️ Utility functions loaded!")
print("Available functions:")
print("  • preview_random_augmentations(person_name) - Show augmentation examples") 
print("  • clean_processing_folders() - Clean all processed data")
print("  • export_face_encodings() - Export encodings for training")

# 🎯 Quick Actions

Run these cells for common tasks:

In [ ]:
# QUICK ACTION: Run complete pipeline
def run_complete_pipeline():
    """Run the complete processing pipeline end-to-end."""
    print("🚀 RUNNING COMPLETE PIPELINE")
    print("=" * 50)
    
    # Step 1: Convert images (HEIC → JPG, standardize)
    print("\n1️⃣ Converting images...")
    convert_all_images()
    
    # Step 2: Grayscale + CLAHE preprocessing
    print("\n2️⃣ Preprocessing (grayscale + CLAHE)...")
    run_preprocessing()
    
    # Step 3: Face detection & ROI extraction
    print("\n3️⃣ Extracting face ROIs...")
    extract_faces_from_preprocessed()
    
    # Step 4: Post-process extracted faces
    print("\n4️⃣ Post-processing faces...")
    post_process_extracted_faces()
    
    # Step 5a: Raw image augmentation (preview & illumination evaluation)
    print("\n5️⃣a Augmenting raw images (for evaluation)...")
    run_augmentation()
    
    # Step 5b: Augment face crops for training
    print("\n5️⃣b Augmenting face crops for training...")
    augment_face_crops()
    
    # Step 6: Prepare training data
    print("\n6️⃣ Preparing training dataset...")
    global training_data
    training_data = prepare_training_data()
    
    # Step 7: Train LBPH model
    print("\n7️⃣ Training LBPH model...")
    global lbph_result
    lbph_result = train_lbph_model(training_data)
    
    # Step 8: Generate summary
    print("\n8️⃣ Generating summary...")
    generate_processing_summary()
    
    print("\n🎉 PIPELINE COMPLETE!")
    print("📊 Run plot_evaluation(lbph_result) to see detailed evaluation")
    print("📊 Run compare_augmentation_impact() to compare with/without augmentation")

# Uncomment the line below to run the complete pipeline
# run_complete_pipeline()

print("💡 Uncomment the line above to run the complete pipeline")
print("⚠️  This will process all images in your dataset")


In [ ]:
# QUICK ACTION: Clean old pipeline folders
def clean_old_pipeline_folders():
    """Clean remnants from the old augmentation pipeline"""
    import shutil
    
    old_folders = [
        os.path.join(DATA_FOLDER, "processed"),    # Old face extraction folder
        os.path.join(DATA_FOLDER, "detectedFaces"), # Old detection folder
    ]
    
    print("🧹 CLEANING OLD PIPELINE FOLDERS")
    print("=" * 40)
    
    for folder in old_folders:
        folder_name = os.path.basename(folder)
        if os.path.exists(folder):
            try:
                shutil.rmtree(folder)
                print(f"✅ Deleted old folder: {folder_name}/")
            except Exception as e:
                print(f"❌ Error deleting {folder_name}/: {e}")
        else:
            print(f"⚠️ Not found: {folder_name}/")
    
    print("\n🧹 Cleanup complete!")

# QUICK ACTION: Show current folder status  
def show_folder_status():
    """Show status of all data folders"""
    print("📁 CURRENT FOLDER STATUS")
    print("=" * 50)
    
    # Current workflow folders
    current_folders = [
        ("raw",              RAW_FOLDER,              "📷 Original images (input)"),
        ("augmented",        AUGMENTED_FOLDER,         "🔄 Augmented images (Member 3)"),
        ("converted",        CONVERTED_FOLDER,         "🔄 Standardized JPGs"),
        ("preprocessed",     PREPROCESSED_FOLDER,      "🔄 Grayscale + CLAHE"),
        ("faces_detected",   FACES_DETECTED_FOLDER,    "👤 Extracted face ROIs"),
        ("final_processed",  FINAL_PROCESSED_FOLDER,   "⚡ Final processed faces"),
        ("preview",          PREVIEW_FOLDER,            "👁️ Preview & evaluation"),
        ("models",           MODELS_FOLDER,             "🧠 Trained models"),
    ]
    
    print("\n📂 ALL PIPELINE FOLDERS:")
    for name, path, description in current_folders:
        exists = "✅" if os.path.exists(path) else "❌"
        files_count = 0
        if os.path.exists(path):
            try:
                # Count files recursively (including subfolders)
                for root, dirs, files in os.walk(path):
                    files_count += len([f for f in files if not f.startswith('.')])
            except:
                files_count = 0
        print(f"  {exists} {name:17} {files_count:5d} files  {description}")
    
    # Check for old pipeline folders that can be cleaned
    old_folders = ["processed", "detectedFaces"]
    old_found = []
    for name in old_folders:
        path = os.path.join(DATA_FOLDER, name)
        if os.path.exists(path):
            old_found.append(name)
    
    if old_found:
        print(f"\n⚠️ Old pipeline folders found: {', '.join(old_found)}")
        print("   Run clean_old_pipeline_folders() to remove them.")

# Show current status
show_folder_status()

# ✅ Pipeline Complete!

Your image processing pipeline is now ready! Here's what you have:

## 📊 Pipeline Summary
| Step | Description | Input → Output | Owner |
|------|-------------|----------------|-------|
| 1 | **Image Conversion** | `raw/` → `converted/` (HEIC→JPG, standardize) | Member 1 |
| 2 | **Grayscale & CLAHE** | `converted/` → `preprocessed/` | Member 1 |
| 3 | **Face Detection & ROI** | `preprocessed/` → `faces_detected/` (128×128) | Member 1 |
| 4 | **Post-Processing** | `faces_detected/` → `final_processed/` (CLAHE on crops) | Member 1 |
| 5a | **Raw Image Augmentation** | `raw/` → `augmented/` (evaluation) | Member 3 |
| 5b | **Face Crop Augmentation** | `final_processed/` → `augmented/` (~100/person) | Member 3 |
| 6 | **Training Data Prep** | `augmented/` → NumPy arrays (80/20 split) | Member 3 |
| 7 | **LBPH Training** | Training arrays → `models/lbph_model.yml` | Member 3 |

## 📊 Evaluation Outputs
| Evaluation | Output |
|-----------|--------|
| Augmentation Preview | `preview/augmentation_samples/` |
| Illumination Robustness | `preview/evaluation_results/illumination_robustness_evaluation.png` |
| LBPH Confusion Matrix | `preview/evaluation_results/lbph_evaluation.png` |
| With vs Without Aug | `preview/evaluation_results/augmentation_impact_comparison.png` |

## 🚀 Next Steps
1. Run the complete pipeline using the quick action cell above
2. Review LBPH evaluation results in `data/preview/evaluation_results/`
3. Compare augmentation impact with `compare_augmentation_impact()`
4. Integrate trained model (`models/lbph_model.yml`) into the attendance application
5. Implement live face recognition using the trained LBPH model

## 💡 Tips
- Each person's ~16-20 face crops → ~100 augmented training images
- The pipeline preserves original images while creating organized copies
- LBPH model is saved to `models/` with a `label_map.txt` for person lookup
- Run `compare_augmentation_impact()` to see the value of augmentation quantitatively

Your facial recognition attendance system is now ready! 🎉
